# Annotation des qualificatifs par analyse syntaxique (spaCy)


## 1. Import des librairies

In [1]:
import re
import pandas as pd
import spacy
import json
import unicodedata
from pathlib import Path
from collections import defaultdict
from collections import Counter
import ast

c:\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


## 2. Import des données et des modèles

In [2]:
FICHIER_ENTREE = "voies_par_longueur_annotation_cotationfr.csv"

# Fichier créé pour le test contrôlé.
FICHIER_SORTIE_TEST = "annotation_spacy_test_50_occurrences.csv"

# Fichier de sortie final, utilisé lorsque SAMPLE_SIZE = None.
FICHIER_SORTIE_COMPLET = "annotation_spacy_occurrences_detaillees.csv"

In [3]:
def charger_modele_spacy(nom_modele):
    """
    Charge le modèle spaCy et vérifie que l'analyse syntaxique est disponible.
    """
    try:
        nlp = spacy.load(nom_modele)
    except OSError as exc:
        raise OSError(
            f"Le modèle spaCy '{nom_modele}' n'est pas installé.\n"
            "Dans Google Colab, exécute cette commande dans une cellule :\n"
            f"    !python -m spacy download {nom_modele}\n"
            "Puis relance le script."
        ) from exc

    if not nlp.has_pipe("parser"):
        raise RuntimeError(
            f"Le modèle '{nom_modele}' ne contient pas de composant parser. "
            "Un modèle avec analyse syntaxique est nécessaire pour proposer "
            "les associations entre termes et qualificatifs."
        )

    return nlp


def preparer_echantillon(df, taille, graine):
    """
    Sélectionne un échantillon reproductible de lignes du CSV.

    Le choix est fait au niveau des descriptions sources, avant la détection
    des termes. Ainsi, les descriptions sans terme sont aussi représentées dans
    le contrôle de couverture, même si elles ne donnent pas de ligne de sortie.
    """
    if taille is None:
        return df.copy()

    if taille <= 0:
        raise ValueError("SAMPLE_SIZE doit être un entier positif ou None.")

    n = min(taille, len(df))
    return df.sample(n=n, random_state=graine).sort_index().copy()


## 3. Définition des paramètres

In [4]:
# 50 = traiter 50 descriptions sélectionnées aléatoirement de façon reproductible.
# None = traiter toutes les descriptions.
SAMPLE_SIZE = None
RANDOM_SEED = 42

# Nombre de tokens spaCy avant et après le terme détecté.
FENETRE_TOKENS = 3

# Modèle français avec analyse syntaxique.
MODELE_SPACY = "fr_core_news_md"

## 4. Dictionnaire des formes à identifier

In [5]:
# Les clés sont les termes canoniques. Les listes contiennent les formes à détecter dans le texte. Le dictionnaire est volontairement conservé tel quefourni ; 
# le script n'ajoute pas de termes de son propre chef.

FORMES = {
    "dévers": [
        "dévers", "déversante", "déversant", "déversantes", "déversants",
        "déversé", "déversée", "déversées", "déversés", "toit", "toits"
    ],
    "surplomb": [
        "surplombant", "surplombante", "surplombé", "surplomb", "surplombs",
        "bombé", "bombés", "bombée", "bombées"
    ],
    "dalle": ["dalle", "dalles", "adhérence", "adhérences"],
    "faille": ["faille", "failles"],
    "épaule": ["épaule", "épaules", "épaulement", "épaulements"],
    "dièdre": ["dièdre", "dièdres"],
    "cheminée": ["cheminée", "cheminées"],
    "vire": ["vire", "vires", "virette", "virettes"],
    "terrasse": ["terrasse", "terrasses", "plateforme", "plateformes"],
    "fissure": ["fissure", "fissures"],
    "traversée": [
        "traversée", "traversées", "traverser", "traverse", "traversant"
    ],
    "réglette": ["réglette", "réglettes"],
    "bac": [
        "bac", "grosse prise", "grosses prises", "bonne prise", "bonnes prises"
    ],
    "plat": ["plat", "plats", "replat", "replats"],
    "pince": ["pince", "pinces"],
    "mono_doigt": ["mono-doigt", "monodoigt", "mono-doigts", "monodoigts"],
    "bi_doigt": ["bi-doigt", "bidoigt", "bi-doigts", "bidoigts"],
    "inversee": [
        "inversée", "inversées", "inversé", "inversés", "inverse", "inverses"
    ],
    "crochet": ["crochet", "crochets"],
    "trou": ["trou", "trous"],
    "goutte_eau": ["goutte d'eau", "gouttes d'eau"],
    "petite_prise": ["petite prise", "petites prises"],
    "écailles": ["écaille", "écailles"],
    "arête": ["arête", "arêtes"],
    "pilier": ["pilier", "piliers"],
    "couloir": ["couloir", "couloirs"],
    "cannelure": [
        "cannelure", "cannelures", "cannelé", "cannelée", "cannelés",
        "cannelées", "canelée", "canelé"
    ],
    "éperon": ["éperon", "éperons"],
    "contrefort": ["contrefort", "contreforts"],
    "corniche": ["corniche", "corniches"],
    "fissure-dièdre": [
        "fissure-dièdre", "fissures-dièdre", "fissure-dièdres",
        "fissures-dièdres"
    ],
}

## 5. Définition des fonctions utiles

In [6]:
def normaliser_texte(texte):
    """
    Normalise un texte pour comparer les formes sans tenir compte de la casse.

    Les accents sont conservés : on ne confond donc pas arbitrairement deux
    graphies différentes. Les apostrophes typographiques sont harmonisées.
    """
    texte = str(texte).replace("’", "'").replace("‘", "'")
    return texte.casefold()


def compiler_motif_formes(dictionnaire):
    """
    Construit un motif regex qui repère toutes les formes du dictionnaire.

    Les formes longues sont placées en premier afin que, lorsqu'elles commencent
    au même endroit, une expression comme « fissure-dièdre » soit reconnue
    avant une forme plus courte.

    Les limites Unicode empêchent par exemple de repérer « trou » dans
    « trousses » ou « bac » au milieu d'un autre mot.
    """
    formes_aplaties = []

    for terme_canonique, liste_formes in dictionnaire.items():
        for forme in liste_formes:
            formes_aplaties.append((terme_canonique, forme))

    # D'abord les formes les plus longues, puis ordre alphabétique stable.
    formes_aplaties.sort(key=lambda item: (-len(item[1]), item[1].casefold()))

    alternatives = []
    for _, forme in formes_aplaties:
        # Les apostrophes typographiques sont traitées comme des apostrophes.
        forme_regex = re.escape(forme).replace("’", "['’]").replace("'", "['’]")
        alternatives.append(forme_regex)

    motif = re.compile(
        r"(?<!\w)(?P<forme>" + "|".join(alternatives) + r")(?!\w)",
        flags=re.IGNORECASE | re.UNICODE
    )
    return motif


def construire_index_formes(dictionnaire):
    """
    Crée une table de correspondance forme -> termes canoniques.

    Une forme peut théoriquement appartenir à plusieurs clés ; on conserve alors
    toutes les clés et on signalera cette ambiguïté dans le résultat.
    """
    index = {}

    for terme_canonique, liste_formes in dictionnaire.items():
        for forme in liste_formes:
            cle = normaliser_texte(forme)
            index.setdefault(cle, [])
            if terme_canonique not in index[cle]:
                index[cle].append(terme_canonique)

    return index


def trouver_colonne_id(df):
    """Retourne le nom de la première colonne d'identifiant connue, si présente."""
    candidats = ["document_id", "id", "ID", "route_id"]
    for colonne in candidats:
        if colonne in df.columns:
            return colonne
    return None


def extraire_occurrences(texte, motif, index_formes):
    """
    Repère les occurrences lexicales dans le texte original.

    Retourne une occurrence par correspondance regex. Les formes détectées
    restent distinctes ; les chevauchements éventuels seront signalés ensuite.
    """
    occurrences = []

    if not isinstance(texte, str) or not texte.strip():
        return occurrences

    for match in motif.finditer(texte):
        forme = match.group("forme")
        cle_forme = normaliser_texte(forme)
        termes = index_formes.get(cle_forme, [])

        if not termes:
            # Cas de sécurité : la forme devrait toujours être dans l'index.
            continue

        occurrences.append({
            "forme_detectee": forme,
            "termes_canoniques": termes,
            "position_debut": match.start(),
            "position_fin": match.end(),
        })

    return occurrences


def indices_tokens_pour_occurrence(doc, debut_caractere, fin_caractere):
    """
    Associe une occurrence repérée dans le texte brut aux tokens spaCy.

    Les offsets de caractères permettent de garder la bonne occurrence même
    lorsque la forme apparaît plusieurs fois dans la description.
    """
    tokens = [
        token for token in doc
        if token.idx < fin_caractere
        and token.idx + len(token.text) > debut_caractere
    ]

    if not tokens:
        return None, None

    return tokens[0].i, tokens[-1].i


def obtenir_fenetre(doc, debut_token, fin_token, taille=5):
    """
    Retourne le contexte constitué de N tokens avant et N tokens après.

    Le terme repéré lui-même n'est pas inclus dans les cinq tokens de chaque
    côté ; l'expression entière est affichée au centre de la fenêtre.
    """
    debut_fenetre = max(0, debut_token - taille)
    fin_fenetre = min(len(doc), fin_token + taille + 1)

    tokens = list(doc[debut_fenetre:fin_fenetre])
    return " ".join(token.text for token in tokens)


def phrase_autour_occurrence(doc, debut_token, fin_token):
    """
    Retourne la phrase spaCy contenant l'occurrence, lorsque disponible.
    """
    token = doc[debut_token]
    try:
        return token.sent.text.strip()
    except Exception:
        return " ".join(t.text for t in doc[max(0, debut_token - 8):fin_token + 9])



## 6. Extraction

In [7]:
def token_est_qualificatif(token):
    """
    Indique si un token peut être un qualificatif descriptif.
    """
    lemmes_exclus = {
        "typer", "nouveau", "partir", "continuer", "traverser",
        "suivre", "emprunter", "pousser", "caractériser",
        "suivant", "supérieur", "gauche", "droite", "long",
        "rocheux", "bord",
    }

    if token.lemma_.casefold() in lemmes_exclus:
        return False

    if token.pos_ == "ADJ":
        return True

    # spaCy étiquette parfois en NOUN des adjectifs rares
    # (« végétative », « Jolies »). Si la relation syntaxique dit
    # bien « modificateur adjectival », on fait confiance à la relation.
    if token.pos_ == "NOUN" and token.dep_ == "amod":
        return True

    # Selon l'analyse du modèle français, certains participes (« couché »,
    # « soutenue ») peuvent être étiquetés VERB plutôt que ADJ.
    if token.pos_ in {"VERB", "AUX"}:
        forme_verbale = token.morph.get("VerbForm")
        if "Part" in forme_verbale and token.dep_ in {
            "amod", "acl", "acl:relcl", "ROOT", "conj"
        }:
            return True

    return False


NOMS_QUALIFICATIFS = {
    "trou", "trous", "adhérence", "adhérences", "dülfer", "dulfer",
    "cannelure", "cannelures", "bac", "bacs", "écaille", "écailles",
    "fissure", "fissures", "réglette", "réglettes", "végétation",
}

def qualificatif_nominal(token, indices_terme):
    """
    Noms d'une liste blanche qui qualifient le terme :
    « dalle à trous », « fissure dülfer », « dalle en adhérence ».
    Les autres noms (gauche, piton, mètres, bord...) ne sont jamais
    des qualificatifs : ils produisaient l'essentiel du bruit.
    """
    if token.lemma_.casefold() not in NOMS_QUALIFICATIFS:
        return False
    if token.head.i in indices_terme:
        return True
    if token.head.pos_ == "ADP" and token.head.head.i in indices_terme:
        return True
    return False


def trouver_qualificatifs(doc, debut_token, fin_token, taille=3):
    """
    Trouve des candidats en privilégiant les relations syntaxiques directes.

    Améliorations :
    - accepte certains participes adjectivaux, comme « couché » et « soutenue » ;
    - regroupe les intensifieurs/comparatifs (« peu raide », « plus soutenue ») ;
    - ajoute un repli de proximité très local pour les adjectifs directement
      avant/après le terme, même si le parseur a mal analysé la relation ;
    - traite « qui est plus propre » comme une description possible du nom
      immédiatement antérieur ;
    - n'utilise pas une simple proximité pour associer un adjectif à un terme
      verbal comme « traverser » : cela évite notamment « gros » -> « traversée »
      dans « deux gros becquets et traverser ».
    """
    debut_fenetre = max(0, debut_token - taille)
    fin_fenetre = min(len(doc), fin_token + taille + 1)

    # Extension ciblée pour une relative du type « l'arête qui est plus propre ».
    fin_etendue = min(len(doc), fin_token + 7)
    tokens_fenetre = list(doc[debut_fenetre:max(fin_fenetre, fin_etendue)])
    tokens_terme = list(doc[debut_token:fin_token + 1])
    indices_terme = {t.i for t in tokens_terme}
    cible_est_verbe_traverser = any(
        t.pos_ in {"VERB", "AUX"} and t.lemma_.casefold() == "traverser"
        for t in tokens_terme
    )

    candidats_forts = []
    candidats_fenetre = []
    vus = set()

    def ajouter(item, direct):
        cle = (item["position_debut"], item["position_fin"], item["texte"])
        if cle in vus:
            return
        vus.add(cle)
        item["lien_syntaxique_direct"] = bool(direct)
        item.setdefault("admissible_attribution", bool(direct))
        (candidats_forts if direct else candidats_fenetre).append(item)


    for token in tokens_fenetre:
        if token.i in indices_terme or token.is_punct or token.like_num:
            continue
        if not (
            token_est_qualificatif(token)
            or qualificatif_nominal(token, indices_terme)
        ):
            continue

        # Évite les qualificatifs éloignés inclus uniquement par l'extension
        # spéciale « qui est plus ... », sauf si cette structure est présente.
        distance = min(
            abs(token.i - debut_token),
            abs(token.i - fin_token)
        )
        relative_plus_propre = (
            token.lemma_.casefold() == "propre"
            and token.i > fin_token
            and any(t.lower_ == "qui" for t in doc[fin_token + 1:token.i])
            and any(t.lower_ == "est" for t in doc[fin_token + 1:token.i])
            and any(t.lower_ == "plus" for t in doc[fin_token + 1:token.i])
        )
        if distance > taille and not relative_plus_propre:
            continue
        
        direct_syntaxe = (
            token.head.i in indices_terme
            or any(t.head.i == token.i for t in tokens_terme)
            or (
                not cible_est_verbe_traverser
                and any(
                    t.dep_ == "conj" and token.head.i == t.head.i
                    for t in tokens_terme
                )
            )
        )

        # Repli local : adjectif immédiatement avant/après le terme, par exemple
        # « franche traversée », « jolies adhérences », « vire confortable »,
        # « dièdre couché ». On ne l'applique pas à une cible verbale « traverser ».
        adjacent_local = (
            distance <= taille
            and not cible_est_verbe_traverser
            and token.pos_ in {"ADJ", "VERB"}
        )

        # Un adjectif qui dépend d'un autre nom (ex. « gros becquets ») ne doit
        # pas être récupéré comme qualificatif d'un terme voisin par proximité.
        tete_autre_nom = (
            token.head.pos_ in {"NOUN", "PROPN"}
            and token.head.i not in indices_terme
            and token.head.i != token.i
        )
        if tete_autre_nom and not direct_syntaxe:
            adjacent_local = False

        # Les liens syntaxiques réels restent dans la colonne syntaxe. Les
        # associations reposant sur la seule proximité restent des candidats
        # de fenêtre ; la règle explicite « qui est plus propre » est conservée
        # comme association forte car elle identifie une relative descriptive.
        direct = direct_syntaxe or relative_plus_propre
        candidat_local = adjacent_local and not direct

        # Regroupe les adverbes qui modifient l'adjectif :
        # « peu raide », « plus soutenue », « très facile ».
        modificateurs = [
            child for child in token.children
            if child.pos_ == "ADV" and child.dep_ in {"advmod", "fixed"}
        ]

        # Repli lexical ciblé : dans certains modèles, « plus » n'est pas
        # rattaché comme advmod à « soutenue/propre/raide ».
        if not modificateurs and token.i > 0 and doc[token.i - 1].lower_ in {
            "plus", "moins", "très", "peu", "assez", "trop"
        }:
            modificateurs = [doc[token.i - 1]]

        modificateurs.sort(key=lambda t: t.i)
        debut = min([token.idx] + [t.idx for t in modificateurs])
        fin = max(
            [token.idx + len(token.text)]
            + [t.idx + len(t.text) for t in modificateurs]
        )
        expression = doc.char_span(debut, fin, alignment_mode="expand")
        texte_expr = expression.text if expression is not None else token.text
        lemme_expr = " ".join(
            [t.lemma_ for t in modificateurs] + [token.lemma_]
        ).strip()

        ajouter({
            "texte": texte_expr,
            "lemme": lemme_expr,
            "relation": (
                f"groupe_adjectival:{token.dep_}"
                if direct else "proximite_locale"
            ),
            "position_debut": debut,
            "position_fin": fin,
            "exception_coordination": any(
                t.dep_ == "conj" and t.head.i == token.head.i
                for t in tokens_terme
            ) or any(child.dep_ == "conj" for child in token.head.children),
            # Les candidats de fenêtre restent visibles pour l'audit, mais ne
            # peuvent être retenus que s'ils sont très proches ou explicitement
            # reconnus par une règle de structure.
            "admissible_attribution": bool(
                direct_syntaxe or adjacent_local or relative_plus_propre
            ),
        }, direct)
    # Expressions descriptives fréquentes qui ne sont pas des adjectifs :
    # « traversée en ascendance », y compris « en ascendance à droite ».
    if any("travers" in t.lemma_.casefold() or "travers" in t.text.casefold()
           for t in tokens_terme):
        limite = min(fin_token + 6, len(doc) - 1)
        for i in range(debut_token, limite):
            if (
                doc[i].lower_ == "en"
                and i + 1 < len(doc)
                and "ascendance" in (
                    doc[i + 1].lemma_.casefold() + " " + doc[i + 1].text.casefold()
                )
            ):
                ajouter({
                    "texte": f"{doc[i].text} {doc[i + 1].text}",
                    "lemme": "en ascendance",
                    "relation": "expression_descriptive_connue",
                    "position_debut": doc[i].idx,
                    "position_fin": doc[i + 1].idx + len(doc[i + 1].text),
                    "exception_coordination": False,
                    "admissible_attribution": True,
                }, True)
                break

    return candidats_forts, candidats_fenetre


def resoudre_exclusivite_qualificatifs(resultats):
    """Attribue chaque qualificatif candidat à un seul terme par défaut.

    Les candidats restent tous visibles dans les colonnes de contrôle. La
    colonne `qualificatifs_retenus` contient uniquement l'attribution finale.
    Si le même segment candidat apparaît pour plusieurs termes, la priorité va
    au lien syntaxique direct, puis à la distance en tokens la plus faible.
    """
    import json

    candidats_par_cle = {}
    candidats_par_ligne = []
    for idx, resultat in enumerate(resultats):
        forts = json.loads(resultat.get("qualificatifs_syntaxe", "[]") or "[]")
        fenetre = json.loads(resultat.get("qualificatifs_fenetre", "[]") or "[]")
        candidats = []
        for item in forts:
            candidats.append((item, True))
        for item in fenetre:
            candidats.append((item, False))
        candidats_par_ligne.append(candidats)
        cible = (resultat["position_debut"] + resultat["position_fin"]) / 2
        for item, direct in candidats:
            # Un candidat faible peut rester visible dans la colonne de fenêtre
            # sans être attribué automatiquement à un terme.
            if not item.get("admissible_attribution", True):
                continue
            cle = (item.get("position_debut"), item.get("position_fin"), item.get("texte"))
            distance = abs((item.get("position_debut", 0) + item.get("position_fin", 0)) / 2 - cible)
            score = (0 if direct or item.get("lien_syntaxique_direct") else 1, distance)
            candidats_par_cle.setdefault(cle, []).append((score, idx, item, direct))

    retenus = [[] for _ in resultats]
    conflits = [False for _ in resultats]
    for cle, propositions in candidats_par_cle.items():
        # Les exceptions de coordination ne sont pas généralisées : si spaCy
        # ne fournit pas de lien direct clair, on garde un seul propriétaire.
        propositions.sort(key=lambda x: (x[0], x[1]))
        # Partage autorisé uniquement si l'analyse signale explicitement une
        # coordination et que les propositions concurrentes sont syntaxiques.
        partage_coordonne = (
            len(propositions) > 1
            and all(p[0][0] == 0 for p in propositions)
            and any(p[2].get("exception_coordination", False) for p in propositions)
        )
        if partage_coordonne:
            for proposition in propositions:
                retenus[proposition[1]].append(proposition[2])
        else:
            meilleur = propositions[0]
            retenus[meilleur[1]].append(meilleur[2])
        if len(propositions) > 1 and not partage_coordonne:
            for proposition in propositions:
                conflits[proposition[1]] = True

    for idx, resultat in enumerate(resultats):
        resultat["qualificatifs_retenus"] = json.dumps(
            sorted(retenus[idx], key=lambda x: x.get("position_debut", 0)),
            ensure_ascii=False
        )
        if conflits[idx]:
            resultat["a_verifier"] = True
            motif = resultat.get("motif_verification", "") or ""
            ajout = "un même qualificatif candidat concernait plusieurs termes ; attribution unique effectuée"
            resultat["motif_verification"] = (motif + " ; " if motif else "") + ajout

    return resultats

def detecter_ambiguite(
    occurrence,
    toutes_occurrences,
    qualificatifs_forts,
    qualificatifs_fenetre
):
    """
    Produit des motifs de vérification transparents.

    Les règles sont volontairement prudentes :
    - forme appartenant à plusieurs termes canoniques ;
    - forme adjectivale potentiellement descriptive d'un autre élément ;
    - occurrence chevauchant une autre occurrence ;
    - qualificatifs trouvés uniquement dans la fenêtre, sans lien syntaxique fort ;
    - absence de qualificatif candidat.
    """
    motifs = []

    if len(occurrence["termes_canoniques"]) > 1:
        motifs.append("forme associée à plusieurs termes canoniques")

    debut = occurrence["position_debut"]
    fin = occurrence["position_fin"]

    for autre in toutes_occurrences:
        if autre is occurrence:
            continue
        chevauchement = (
            debut < autre["position_fin"]
            and autre["position_debut"] < fin
        )
        if chevauchement:
            motifs.append(
                "chevauchement avec une autre forme du dictionnaire "
                f"({autre['forme_detectee']})"
            )
            break

    forme_normalisee = normaliser_texte(occurrence["forme_detectee"])
    formes_adjectivales_ambiguës = {
        "cannelé", "cannelée", "cannelés", "cannelées", "canelé", "canelée",
        "surplombant", "surplombante", "déversant", "déversante",
        "déversé", "déversée", "inversé", "inversée"
    }
    if forme_normalisee in formes_adjectivales_ambiguës:
        motifs.append(
            "forme pouvant être un qualificatif d'un autre élément selon le contexte"
        )

    if qualificatifs_fenetre and not qualificatifs_forts:
        motifs.append(
            "qualificatif(s) présent(s) dans la fenêtre sans lien syntaxique fort"
        )

    if not qualificatifs_forts and not qualificatifs_fenetre:
        motifs.append("aucun qualificatif candidat détecté dans la fenêtre")

    return bool(motifs), " ; ".join(dict.fromkeys(motifs))


## 7. Annotation d'une description

In [8]:
def annoter_description(
    texte,
    doc,
    motif_formes,
    index_formes,
    fenetre_tokens=5
):
    """
    Retourne une liste de lignes détaillées : une ligne par occurrence détectée.

    Les candidats sont conservés séparément des associations retenues.
    Une attribution unique est résolue ensuite à l’échelle de la description.
    """
    occurrences = extraire_occurrences(texte, motif_formes, index_formes)
    resultats = []

    for occurrence in occurrences:
        debut_token, fin_token = indices_tokens_pour_occurrence(
            doc,
            occurrence["position_debut"],
            occurrence["position_fin"]
        )

        if debut_token is None:
            # L'occurrence n'a pas pu être alignée sur les tokens spaCy.
            contexte = texte[
                max(0, occurrence["position_debut"] - 40):
                min(len(texte), occurrence["position_fin"] + 40)
            ]
            qualificatifs_forts = []
            qualificatifs_fenetre = []
            phrase = contexte
            erreur_alignement = True
        else:
            contexte = obtenir_fenetre(
                doc, debut_token, fin_token, taille=fenetre_tokens
            )
            phrase = phrase_autour_occurrence(doc, debut_token, fin_token)
            qualificatifs_forts, qualificatifs_fenetre = trouver_qualificatifs(
                doc, debut_token, fin_token, taille=fenetre_tokens
            )
            erreur_alignement = False

        a_verifier, motif_verification = detecter_ambiguite(
            occurrence,
            occurrences,
            qualificatifs_forts,
            qualificatifs_fenetre
        )

        if erreur_alignement:
            a_verifier = True
            motif_verification = (
                motif_verification + " ; " if motif_verification else ""
            ) + "alignement occurrence / tokens spaCy impossible"

        # Une ligne par terme canonique, tout en gardant la forme détectée.
        for terme_canonique in occurrence["termes_canoniques"]:
            resultats.append({
                "terme_canonique": terme_canonique,
                "forme_detectee": occurrence["forme_detectee"],
                "position_debut": occurrence["position_debut"],
                "position_fin": occurrence["position_fin"],
                "fenetre_contexte": contexte,
                "phrase_contexte": phrase,
                "qualificatifs_syntaxe": json.dumps(
                    qualificatifs_forts, ensure_ascii=False
                ),
                "qualificatifs_fenetre": json.dumps(
                    qualificatifs_fenetre, ensure_ascii=False
                ),
                "a_verifier": a_verifier,
                "motif_verification": motif_verification,
            })

    return resultats



## 8. Annotation de toutes les descriptions

In [9]:
def main():
    print("=" * 72)
    print("ANNOTATION SPAcy DES TERMES ET QUALIFICATIFS")
    print("=" * 72)

    chemin_entree = Path(FICHIER_ENTREE)
    if not chemin_entree.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {chemin_entree.resolve()}\n"
            "Place le CSV dans le dossier courant ou adapte FICHIER_ENTREE."
        )

    print(f"\nLecture du fichier : {chemin_entree.resolve()}")
    df = pd.read_csv(chemin_entree, low_memory=False)

    if "description" not in df.columns:
        raise KeyError(
            "La colonne 'description' est introuvable dans le CSV. "
            f"Colonnes disponibles : {list(df.columns)}"
        )

    print(f"Nombre total de lignes source : {len(df):,}")
    print(f"Nombre de colonnes source : {len(df.columns)}")

    colonne_id = trouver_colonne_id(df)
    if colonne_id:
        print(f"Colonne identifiant utilisée : {colonne_id}")
    else:
        print("Aucune colonne ID reconnue ; l'index source sera conservé.")

    # Ajoute une provenance stable : numéro de ligne dans le CSV, à partir de 1.
    # Ce numéro est créé avant l'échantillonnage et reste donc relié au fichier
    # d'origine même si l'on ne traite qu'un sous-ensemble.
    df = df.copy()
    df["source_ligne"] = range(1, len(df) + 1)

    df_test = df.head(SAMPLE_SIZE).copy()

    if SAMPLE_SIZE is None:
        print("Mode : traitement de toutes les descriptions.")
        fichier_sortie = FICHIER_SORTIE_COMPLET
    else:
        print(
            f"Mode test : {len(df_test)} descriptions échantillonnées "
            f"(graine aléatoire = {RANDOM_SEED})."
        )
        fichier_sortie = FICHIER_SORTIE_TEST

    print(f"\nChargement du modèle spaCy : {MODELE_SPACY}")
    nlp = charger_modele_spacy(MODELE_SPACY)

    motif_formes = compiler_motif_formes(FORMES)
    index_formes = construire_index_formes(FORMES)

    lignes_detaillees = []
    nb_descriptions_avec_match = 0
    nb_occurrences = 0
    nb_descriptions_sans_match = 0

    # nlp.pipe est plus efficace que d'appeler nlp() séparément pour chaque
    # description, surtout sur plusieurs dizaines de milliers de lignes.
    descriptions = [
        "" if pd.isna(valeur) else str(valeur)
        for valeur in df_test["description"].tolist()
    ]

    print("\nAnalyse en cours...")
    for position, (index_df, doc) in enumerate(
        zip(df_test.index.tolist(), nlp.pipe(descriptions, batch_size=64)),
        start=1
    ):
        ligne = df_test.loc[index_df]
        texte = descriptions[position - 1]

        resultats = annoter_description(
            texte=texte,
            doc=doc,
            motif_formes=motif_formes,
            index_formes=index_formes,
            fenetre_tokens=FENETRE_TOKENS
        )

        resultats = resoudre_exclusivite_qualificatifs(resultats)

        if resultats:
            nb_descriptions_avec_match += 1
            nb_occurrences += len(resultats)
        else:
            nb_descriptions_sans_match += 1

        for resultat in resultats:
            ligne_sortie = {
                "source_ligne": int(ligne["source_ligne"]),
                "index_dataframe": int(index_df),
                "document_id": (
                    ligne[colonne_id]
                    if colonne_id is not None else None
                ),
                "texte_original": texte,
            }
            ligne_sortie.update(resultat)
            lignes_detaillees.append(ligne_sortie)

        if position % 10 == 0 or position == len(df_test):
            print(
                f"  Progression : {position}/{len(df_test)} descriptions "
                f"| occurrences conservées : {len(lignes_detaillees)}"
            )

    colonnes_sortie = [
        "source_ligne",
        "index_dataframe",
        "document_id",
        "texte_original",
        "terme_canonique",
        "forme_detectee",
        "position_debut",
        "position_fin",
        "fenetre_contexte",
        "phrase_contexte",
        "qualificatifs_syntaxe",
        "qualificatifs_fenetre",
        "qualificatifs_retenus",
        "a_verifier",
        "motif_verification",
    ]

    resultat_df = pd.DataFrame(lignes_detaillees, columns=colonnes_sortie)
    resultat_df.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 72)
    print("BILAN")
    print("=" * 72)
    print(f"Descriptions traitées : {len(df_test):,}")
    print(f"Descriptions avec au moins un terme : {nb_descriptions_avec_match:,}")
    print(f"Descriptions sans terme détecté : {nb_descriptions_sans_match:,}")
    print(f"Lignes détaillées produites : {len(resultat_df):,}")

    if not resultat_df.empty:
        print(
            "Lignes signalées pour vérification : "
            f"{int(resultat_df['a_verifier'].fillna(False).sum()):,}"
        )
        print("\nOccurrences par terme canonique :")
        print(
            resultat_df["terme_canonique"]
            .value_counts()
            .to_string()
        )
    else:
        print(
            "\nAucune occurrence n'a été trouvée dans les descriptions traitées. "
            "Vérifie que le CSV et la colonne 'description' sont les bons."
        )

    print(f"\nFichier détaillé créé : {Path(fichier_sortie).resolve()}")

    if SAMPLE_SIZE is not None:
        print(
            "\nÉtape suivante : ouvre et contrôle le CSV de test. "
            "Si les résultats sont satisfaisants, remplace SAMPLE_SIZE = 50 "
            "par SAMPLE_SIZE = None, puis relance le script pour traiter "
            "toutes les descriptions. Le traitement complet écrira dans "
            f"'{FICHIER_SORTIE_COMPLET}'."
        )


if __name__ == "__main__":
    main()


ANNOTATION SPAcy DES TERMES ET QUALIFICATIFS

Lecture du fichier : C:\Users\lunag\Documents\Etudes\Etudes Sup\CPES_L3\ProjetRecherche\Code\Version19\CodePropre\2.TraitementDonnees\voies_par_longueur_annotation_cotationfr.csv
Nombre total de lignes source : 42,861
Nombre de colonnes source : 36
Colonne identifiant utilisée : document_id
Mode : traitement de toutes les descriptions.

Chargement du modèle spaCy : fr_core_news_md

Analyse en cours...
  Progression : 10/42861 descriptions | occurrences conservées : 19
  Progression : 20/42861 descriptions | occurrences conservées : 33
  Progression : 30/42861 descriptions | occurrences conservées : 44
  Progression : 40/42861 descriptions | occurrences conservées : 63
  Progression : 50/42861 descriptions | occurrences conservées : 72
  Progression : 60/42861 descriptions | occurrences conservées : 88
  Progression : 70/42861 descriptions | occurrences conservées : 109
  Progression : 80/42861 descriptions | occurrences conservées : 124
  P